# Testing -- scroll down for working steady state numerical solutions

In [7]:
import sympy as sp

x = sp.symbols('x')

a, b, d, g = sp.symbols('a b d g')

poly = a*x**5 - (g+b)*x**4 + a*d**4*x -b*d**4

sp.solve(poly, x)

roots = sp.solve(poly, x)
print(roots)

[]


In [ ]:
# substitute numbers
poly_num = poly.subs({
    a: 1.0,
    b: 2.0,
    d: 0.5,
    g: 3.0
})

# numerical roots
roots = sp.nroots(poly_num)
print(roots)

[4.99969995799052, -0.281888382605184 - 0.292238618428867*I, -0.281888382605184 + 0.292238618428867*I, 0.282038403609926 - 0.268520290150634*I, 0.282038403609926 + 0.268520290150634*I]


In [9]:
import numpy as np
from scipy.optimize import fsolve


def steadystates(x):
    """
    Python version of steadystates.m

    x[0] = C_0
    x[1] = C_in
    x[2] = C_SR
    """
    # Unknown variables
    C_0 = x[0]
    C_in = x[1]
    C_SR = x[2]

    # -----------------------------
    # Parameters
    # -----------------------------
    p = 0.35

    V_p = 4.5
    k_p = 0.4
    n = 4

    a_0 = 0.05
    a_1 = 0.25
    a_2 = 1.0

    V_e = 4.5
    k_e = 0.1

    k_IP3R = 5.55
    k_RYR = 5.0

    J_er = 0.1

    gCa = 9.0

    V_m = -50.0
    k_m = 12.0

    V = -60.0

    Fconst = 96485.3329
    R = 8314.0
    T = 310.0

    K_1 = 0.13
    K_5 = 0.082

    y = 0.1

    # -----------------------------
    # Auxiliary variables
    # -----------------------------
    m = 1.0 / (1.0 + np.exp(-(V - V_m) / k_m))

    exp_term = np.exp(-2.0 * V * Fconst / (R * T))
    V_Ca = V * (C_in - C_0 * exp_term) / (1.0 - exp_term)

    I_Ca = gCa * m**2 * V_Ca

    P_RYR = C_in**n / (k_p**n + C_in**n)
    P_IP3R = (p * C_in * (1.0 - y) / ((p + K_1) * (C_in + K_5)))**3

    # -----------------------------
    # Fluxes
    # -----------------------------
    J_PMCA = V_p * C_in**n / (k_p**n + C_in**n)

    J_in = a_0 - a_1 * I_Ca / (2.0 * Fconst) + a_2 * p

    J_SERCA = V_e * C_in**2 / (k_e**2 + C_in**2)

    J_IP3R = k_IP3R * P_IP3R * (C_SR - C_in)
    J_RYR = k_RYR * P_RYR * (C_SR - C_in)
    J_leak = J_er * (C_SR - C_in)

    # -----------------------------
    # Steady-state equations
    # -----------------------------
    F = np.zeros(3)
    F[0] = J_PMCA - J_in
    F[1] = J_in - J_PMCA - J_SERCA + J_IP3R + J_RYR + J_leak
    F[2] = J_SERCA - J_IP3R - J_RYR - J_leak

    return F


def main():
    # Initial guess
    x0 = np.array([100.0, 50.0, 25.0])

    # Solve steady state
    x_ss, info, ier, msg = fsolve(
        steadystates,
        x0,
        xtol=1e-10,
        full_output=True
    )

    fval = steadystates(x_ss)

    # Display results
    print("Steady state:")
    print(x_ss)

    print("\nResiduals:")
    print(fval)

    print("\nExit flag:")
    print(ier)

    print("\nMessage:")
    print(msg)


if __name__ == "__main__":
    main()

Steady state:
[63136.28588981  -751.65377063  -750.97901417]

Residuals:
[ 0.0000000e+00 -3.2018832e-13  3.2018832e-13]

Exit flag:
1

Message:
The solution converged.


In [18]:
import numpy as np
from scipy.optimize import root, fsolve


def get_default_params():
    return {
        # Calcium flux parameters
        "k_PMCA": 0.4,
        "V_Pmax": 4.5,
        "V_s": 4.5,
        "k_s": 0.1,
        "k_leak": 0.1,
        "F": 96485.3329,

        "P": 0.35,

        "g_Ca": 9.0,
        "V_m": -50.0,
        "k_m": 12.0,
        "R": 8314.0,
        "T": 310.0,

        "a_0": 0.05,
        "a_1": 0.25,
        "a_2": 1.0,

        "k_1": 2000.0,
        "k_-1": 260.0,
        "K_1": 0.13,

        "k_2": 1.0,
        "k_-2": 1.05,
        "K_2": 1.05,

        "k_3": 2000.0,
        "k_-3": 1886.0,
        "K_3": 0.943,

        "k_4": 1.0,
        "k_-4": 0.145,
        "K_4": 0.145,

        "k_5": 100.0,
        "k_-5": 8.2,
        "K_5": 0.082,

        # Channel gains
        "k_IP3R": 5.55,
        "k_RyR": 5.0,
        "k_ryr0": 0.0072,
        "k_ryr1": 0.334,
        "k_ryr2": 0.5,
        "k_ryr3": 38.0,

        # Functional parameters
        "n": 4,
        "ns": 2,
        "n2": 3,

        # Initial membrane / extracellular conditions
        "C_0": 2.0,   # treat as constant extracellular Ca
        "V": -60.0,
        "y": 0.0
    }


def m_inf(V, p):
    return 1.0 / (1.0 + np.exp(-(V - p["V_m"]) / p["k_m"]))


def V_Ca(V, C_in, C_0, p):
    F = p["F"]
    R = p["R"]
    T = p["T"]

    exp_term = np.exp(-2.0 * V * F / (R * T))
    denom = 1.0 - exp_term

    if np.abs(denom) < 1e-12:
        return 0.0

    return V * (C_in - C_0 * exp_term) / denom


def I_Ca(V, C_in, C_0, p):
    m = m_inf(V, p)
    Vca = V_Ca(V, C_in, C_0, p)
    return p["g_Ca"] * (m ** 2) * Vca


def J_PMCA(C_in, p):
    return p["V_Pmax"] * (C_in ** p["n"]) / (p["k_PMCA"] ** p["n"] + C_in ** p["n"])


def J_in_Wang(V, C_in, C_0, p):
    return p["a_0"] - p["a_1"] * I_Ca(V, C_in, C_0, p) / (2.0 * p["F"]) + p["a_2"] * p["P"]


def J_SERCA(C_in, p):
    return p["V_s"] * (C_in ** p["ns"]) / (p["k_s"] ** p["ns"] + C_in ** p["ns"])


def P_IP3R(C_in, y, p):
    num = p["P"] * C_in * (1.0 - y)
    den = (p["P"] + p["K_1"]) * (C_in + p["K_5"])
    return (num / den) ** p["n2"]


def dy_dt(y, C_in, p):
    f1 = (
        (p["k_4"] * p["K_2"] * p["K_1"] + p["k_2"] * p["K_4"] * p["P"]) * C_in
        / (p["K_4"] * p["K_2"] * (p["K_1"] + p["P"]))
    )
    f2 = (
        (p["k_2"] * p["P"] + p["k_4"] * p["K_3"])
        / (p["K_3"] + p["P"])
    )
    return f1 * (1.0 - y) - f2 * y


def P_RyR(C_in, C_SR, p):
    activation = p["k_ryr0"] + (p["k_ryr1"] * C_in**3) / (p["k_ryr2"]**3 + C_in**3)
    sr_term = C_SR**4 / (p["k_ryr3"]**4 + C_SR**4)
    return activation * sr_term


def J_IP3R_Wang(C_SR, C_in, y, p):
    return p["k_IP3R"] * P_IP3R(C_in, y, p) * (C_SR - C_in)


def J_RyR_Wang(C_SR, C_in, p):
    return p["k_RyR"] * P_RyR(C_in, C_SR, p) * (C_SR - C_in)


def J_leak(C_SR, C_in, p):
    return p["k_leak"] * (C_SR - C_in)


def residuals(x, p):
    """
    Steady-state equations for:
        x[0] = C_0
        x[1] = C_in
        x[2] = C_SR
    """
    C_0, C_in, C_SR = x

    V = p["V"]
    y = p["y"]

    Jin = J_in_Wang(V, C_in, C_0, p)
    Jpm = J_PMCA(C_in, p)
    Jserca = J_SERCA(C_in, p)

    Jip3r = J_IP3R_Wang(C_SR, C_in, y, p)
    Jryr = J_RyR_Wang(C_SR, C_in, p)
    Jleak = J_leak(C_SR, C_in, p)

    F = np.zeros(3)
    F[0] = Jpm - Jin
    F[1] = Jin - Jpm - Jserca + Jip3r + Jryr + Jleak
    F[2] = Jserca - Jip3r - Jryr - Jleak

    return F


def solve_with_root(x0, p):
    sol = root(lambda x: residuals(x, p), x0, method="hybr")
    return {
        "x": sol.x,
        "success": sol.success,
        "message": sol.message,
        "fun": sol.fun,
        "nfev": sol.nfev,
        "status": sol.status,
    }


def solve_with_fsolve(x0, p):
    x_ss, info, ier, msg = fsolve(
        lambda x: residuals(x, p),
        x0,
        xtol=1e-10,
        full_output=True,
    )

    return {
        "x": x_ss,
        "success": ier == 1,
        "message": msg,
        "fun": residuals(x_ss, p),
        "nfev": info.get("nfev", None),
        "status": ier,
    }


def multi_start_solve(initial_guesses, p, tol=1e-6):
    solutions = []

    for x0 in initial_guesses:
        out = solve_with_root(np.array(x0, dtype=float), p)

        if not out["success"]:
            continue

        x = out["x"]

        is_new = True
        for s in solutions:
            if np.linalg.norm(x - s["x"]) < tol:
                is_new = False
                break

        if is_new:
            solutions.append(out)

    return solutions


if __name__ == "__main__":
    params = get_default_params()

    # Single solve
    x0 = np.array([2.0, 0.112, 24.0])
    out = solve_with_root(x0, params)

    print("Single solve")
    print("Success:", out["success"])
    print("Message:", out["message"])
    print("Steady state:", out["x"])
    print("Residuals:", out["fun"])
    print()

    # Multi-start search
    guesses = [
        [2.0, 0.05, 10.0],
        [2.0, 0.10, 20.0],
        [2.0, 0.20, 24.0],
        [2.0, 0.50, 30.0],
        [2.0, 1.00, 40.0],
    ]

    sols = multi_start_solve(guesses, params)

    print("Multi-start solutions")
    for i, s in enumerate(sols, 1):
        print(f"Solution {i}:")
        print("  x =", s["x"])
        print("  residuals =", s["fun"])
        print("  message =", s["message"])
        print()

Single solve
Success: True
Message: The solution converged.
Steady state: [-6.16045554e+03 -5.81473909e-03  1.47287561e-01]
Residuals: [ 1.30634244e-15  1.41683540e-12 -1.41814165e-12]

Multi-start solutions
Solution 1:
  x = [-6.16045732e+03  4.58762327e-03  9.87955301e-02]
  residuals = [-6.64326882e-17  7.17887555e-13 -7.17821635e-13]
  message = The solution converged.

Solution 2:
  x = [-6.15801991e+03  3.08066911e-02  2.74283486e+00]
  residuals = [-2.57514008e-14  1.01179065e-11 -1.00921493e-11]
  message = The solution converged.

Solution 3:
  x = [1.6296388  0.22356844 4.20111513]
  residuals = [-2.09277040e-14  9.82602888e-13 -9.61730695e-13]
  message = The solution converged.

Solution 4:
  x = [1.59791618 0.22356812 4.20111698]
  residuals = [ 0.0000000e+00  8.8817842e-16 -8.8817842e-16]
  message = The solution converged.

Solution 5:
  x = [6.31435292e+04 6.15025279e+00 8.22193889e+00]
  residuals = [ 7.28306304e-14  1.24267541e-11 -1.24995847e-11]
  message = The solu

# Working solution - V constant

In [ ]:
import numpy as np
from scipy.optimize import root, fsolve, least_squares

def get_default_params():
    return {
    # Calcium flux parameters
    "k_PMCA": 0.4,      # Wang extrusion rate 
    "V_Pmax": 4.5,      # Wang PMCA pump max rate 
    "V_s": 4.5,         # Wang/Lytton SR uptake rate max for SERCA pump between 0.1 and 0.5
    "k_s": 0.1,         # Wang/Lytton SR uptake affinity for SERCA pump between 0.1 and 0.5 
    "k_leak": 0.1,      # Wang SR leak
    "F": 96485.3329,    # Physical Faraday's constant 

    "P": 0.35, # IP3 production proxy by agonist 

    "gamma": 5.5,
    "delta": 0.05,

    "g_Ca": 9.0,        # Wang nS mM^-1 
    "V_m": -50.0,       # Wang mV 
    "k_m": 12.0,        # Wang mV 
    "R": 8314,         # Physical mJ/(mol K)
    "T": 310.0,          # Wang K (37°C) 

    "a_0": 0.05, # Wang
    "a_1": 0.25, # Wang
    "a_2": 1, # Wang

    "k_1": 2000, # Wang,. DeY 
    "k_-1": 260, # Wang, DeY
    "K_1": 0.13, # Wang, DeY - derived 
    "k_2": 1, # Wang, DeY
    "k_-2": 1.05, # Wang, DeY
    "K_2": 1.05, # Wang, DeY - derived
    "k_3": 2000, # Wang, DeY
    "k_-3": 1886, # Wang, DeY
    "K_3": 0.943, # Wang, DeY - derived
    "k_4": 1, # Wang, DeY
    "k_-4": 0.145, # Wang, DeY
    "K_4": 0.145, # Wang, DeY - derived
    "k_5": 100, # Wang, DeY
    "k_-5": 8.2, # Wang, DeY
    "K_5": 0.082, # Wang, DeY - derived

    # Channel gains
    "k_IP3R": 5.55, #Wang
    "k_RyR": 5.0, #Wang 
    "k_ryr0": 0.0072, # Wang and Friel RyR opening rate
    "k_ryr1": 0.334, # Wang and Friel and Shannon RyR closing rate
    "k_ryr2": 0.5, # Wang and Friel and Shannon RyR activation affinity
    "k_ryr3": 38.0, # Wang and Shannon RyR inactivation affinity

    # Voltage parameters
    "c_m": 1.0, # Lata
    "I_stim": 0.1175, # Lata

    # Contraction
    "alpha": 3.0, #Lata - uterine
    "beta": 0.001, #Lata - uterine
    "n_F": 4, # Lata - uterine

    # Functional parameters
    "n": 4, # Wang - Hill coefficient for SERCA channel activation 1, 2 or 4
    "ns": 2, # Wang and Lytton
    "n2": 3 # Wang 3 or 5 
    }
# -----------------------------
# Flux definitions
# -----------------------------

def J_in_Wang(V, Ca_in, Ca_0, p):
    return p["a_0"]-p["a_1"]*I_Ca(V, Ca_in, Ca_0, p)/(2*p["F"]) +p["a_2"]*p["P"]  

def m_inf(V, p):
    return 1.0 / (1.0 + np.exp(-(V - p["V_m"]) / p["k_m"]))

def V_Ca(V, Ca_in, Ca_0, p):
    F = p["F"]
    R = p["R"]
    T = p["T"]

    exp_term = np.exp(-2.0 * V * F / (R * T))

    denom = 1.0 - exp_term
    if np.abs(denom) < 1e-8:
        return 1e-8

    return V * (Ca_in - Ca_0 * exp_term) / denom

def I_Ca(V, Ca_in, Ca_0, p):
    m = m_inf(V, p)
    Vca = V_Ca(V, Ca_in, Ca_0, p)
    return p["g_Ca"] * (m**2) * Vca

def J_PMCA_Hill(Ca_in, p):
    return p["V_Pmax"] * (Ca_in**p["n"]) / (p["k_PMCA"]**p["n"] + Ca_in**p["n"]) # - Ca_0**p["n"]

def J_SERCA_Hill(Ca_in, p):
    return p["V_s"] * (Ca_in**p["ns"]) / (p["k_s"]**p["ns"] + Ca_in**p["ns"]) # - Ca_0**p["n"]

def J_leak(Ca_SR, Ca_in, p):
    return p["k_leak"] * (Ca_SR - Ca_in)

def J_IP3R_Wang(Ca_SR, Ca_in, y_g, p):
    return p["k_IP3R"] * P_IP3R(Ca_in, y_g, p) * (Ca_SR - Ca_in)

def P_IP3R(Ca_in, y, p):
    num = p["P"] * Ca_in * (1 - y)
    den = (p["P"] + p["K_1"]) * (Ca_in + p["K_5"])
    return (num / den) ** 3

def dy_dt(y, p, Ca_in):
    f1 = (p["k_-4"] * p["K_2"] * p["K_1"] + p["k_-2"] * p["K_4"] * p["P"]) * Ca_in / (p["K_4"] * p["K_2"] * (p["K_1"] + p["P"]))
    f2 = (p["k_-2"] * p["P"] + p["k_-4"] * p["K_3"]) / (p["K_3"] + p["P"])
    return f1 * (1.0 - y) - f2 * y

def J_RyR_Hill(Ca_SR, Ca_in, p):
    return p["k_RyR"] * (Ca_SR**p["n"]/(p["K_RyR"]**p["n"]+Ca_SR**p["n"])) * (Ca_SR - Ca_in)

def J_RyR_Wang(Ca_SR, Ca_in, p):
    return p["k_RyR"] * P_RyR(Ca_in, Ca_SR, p) * (Ca_SR - Ca_in)

def P_RyR(Ca_in, Ca_SR, p):
    # CICR activation term (cytosolic Ca)
    activation = (
        p["k_ryr0"]
        + (p["k_ryr1"] * Ca_in**3) / (p["k_ryr2"]**3 + Ca_in**3)
    )

    # SR load dependence
    sr_term = Ca_SR**4 / (p["k_ryr3"]**4 + Ca_SR**4)

    return activation * sr_term

def residuals(x, p):
    """
    Steady-state equations for:
        x[0] = C_0
        x[1] = C_in
        x[2] = C_SR
    """
    C_0, C_in, C_SR, y = x
    V = -60

    Jin = p["delta"]*J_in_Wang(V, C_in, C_0, p)
    Jpm = p["delta"]*J_PMCA_Hill(C_in, p)
    Jserca = J_SERCA_Hill(C_in, p)
    Jip3r = J_IP3R_Wang(C_SR, C_in, y, p)
    Jryr = J_RyR_Wang(C_SR, C_in, p)
    Jleak = J_leak(C_SR, C_in, p)

    F = np.zeros(4)
    F[0] = Jpm - Jin
    F[1] = Jin - Jpm - Jserca + Jip3r + Jryr + Jleak
    F[2] = p["gamma"]*(Jserca - Jip3r - Jryr - Jleak)
    # Jserca - Jip3r - Jryr - Jleak
    # F[3] = (C_0 - C_in) / p["c_m"]
    # F[3] = ((p["k_-4"] * p["K_2"] * p["K_1"] + p["k_-2"] * p["K_4"] * p["P"]) * C_in / (p["K_4"] * p["K_2"] * (p["K_1"] + p["P"])))/((p["k_-4"] * p["K_2"] * p["K_1"] + p["k_-2"] * p["K_4"] * p["P"]) * C_in / (p["K_4"] * p["K_2"] * (p["K_1"] + p["P"]))+(p["k_-2"] * p["P"] + p["k_-4"] * p["K_3"]) / (p["K_3"] + p["P"]))
    f1 = (p["k_-4"] * p["K_2"] * p["K_1"] + p["k_-2"] * p["K_4"] * p["P"]) * C_in / (p["K_4"] * p["K_2"] * (p["K_1"] + p["P"]))
    f2 = (p["k_-2"] * p["P"] + p["k_-4"] * p["K_3"]) / (p["K_3"] + p["P"])
    F[3] = f1 * (1.0 - y) - f2 * y

    return F

def solve_with_root(x0, p):
    # sol = root(lambda x: residuals(x, p), x0, method="hybr")

    sol = least_squares(
        lambda x: residuals(x, params),
        x0,
        bounds=(0, np.inf)
        )
    print(sol.x)
    print(sol.fun)

    return {
        "x": sol.x,
        "success": sol.success,
        "message": sol.message,
        "fun": sol.fun,
        "nfev": sol.nfev,
        "status": sol.status,
    }


def solve_with_fsolve(x0, p):
    x_ss, info, ier, msg = fsolve(
        lambda x: residuals(x, p),
        x0,
        full_output=True,
        xtol=1e-10,
    )

    return {
        "x": x_ss,
        "success": ier == 1,
        "message": msg,
        "fun": residuals(x_ss, p),
        "nfev": info.get("nfev", None),
        "status": ier,
    }


def multi_start_solve(initial_guesses, p, tol=1e-6):
    """
    Try many initial guesses and keep unique converged roots.
    """
    solutions = []

    for x0 in initial_guesses:
        out = solve_with_root(np.array(x0, dtype=float), p)

        if not out["success"]:
            continue

        x = out["x"]

        # Deduplicate by closeness
        is_new = True
        for s in solutions:
            if np.linalg.norm(x - s["x"]) < tol:
                is_new = False
                break

        if is_new:
            solutions.append(out)

    return solutions


if __name__ == "__main__":
    params = get_default_params()

    x0 = np.array([1002, 0.1, 25.0, 0])
    out = solve_with_root(x0, params)

    print("Single solve")
    print("Success:", out["success"])
    print("Message:", out["message"])
    print("Steady state:", out["x"])
    print("Resbiduals:", out["fun"])
    print()

    # # Multiple guesses to look for multiple steady states
    # guesses = [
    #     [0.1, 0.1, 1.0, -30, 1],
    #     [1.0, 0.2, 5.0, -50, 0],
    #     [10.0, 0.5, 20.0, -10, 10],
    #     [100.0, 50.0, 25.0, -60, 0],
    #     [0.01, 0.05, 24.0, -20, 5],
    # ]

    # sols = multi_start_solve(guesses, params)

    # print("Multi-start solutions")
    # for i, s in enumerate(sols, 1):
    #     print(f"Solution {i}:")
    #     print("  x =", s["x"])
    #     print("  residuals =", s["fun"])
    #     print("  message =", s["message"])
    #     print()

[1.00080373e+03 2.33054623e-01 1.22736199e+01 3.74067923e-01]
[ 7.71541130e-11 -8.93030094e-11  6.68179956e-11  9.43689571e-16]
Single solve
Success: True
Message: `gtol` termination condition is satisfied.
Steady state: [1.00080373e+03 2.33054623e-01 1.22736199e+01 3.74067923e-01]
Residuals: [ 7.71541130e-11 -8.93030094e-11  6.68179956e-11  9.43689571e-16]



# Adding in Estrogen in the J function

In [55]:
import numpy as np
from scipy.optimize import root, fsolve, least_squares

def get_default_params():
    return {
    # Calcium flux parameters
    "k_PMCA": 0.4,      # Wang extrusion rate 
    "V_Pmax": 4.5,      # Wang PMCA pump max rate 
    "V_s": 4.5,         # Wang/Lytton SR uptake rate max for SERCA pump between 0.1 and 0.5
    "k_s": 0.1,         # Wang/Lytton SR uptake affinity for SERCA pump between 0.1 and 0.5 
    "k_leak": 0.1,      # Wang SR leak
    "F": 96485.3329,    # Physical Faraday's constant 

    "P": 0.35, # IP3 production proxy by agonist 

    "gamma": 5.5,
    "delta": 0.05,

    "g_Ca": 9.0,        # Wang nS mM^-1 
    "V_m": -50.0,       # Wang mV 
    "k_m": 12.0,        # Wang mV 
    "R": 8314,         # Physical mJ/(mol K)
    "T": 310.0,          # Wang K (37°C) 

    "a_0": 0.05, # Wang
    "a_1": 0.25, # Wang
    "a_2": 1, # Wang

    "k_1": 2000, # Wang,. DeY 
    "k_-1": 260, # Wang, DeY
    "K_1": 0.13, # Wang, DeY - derived 
    "k_2": 1, # Wang, DeY
    "k_-2": 1.05, # Wang, DeY
    "K_2": 1.05, # Wang, DeY - derived
    "k_3": 2000, # Wang, DeY
    "k_-3": 1886, # Wang, DeY
    "K_3": 0.943, # Wang, DeY - derived
    "k_4": 1, # Wang, DeY
    "k_-4": 0.145, # Wang, DeY
    "K_4": 0.145, # Wang, DeY - derived
    "k_5": 100, # Wang, DeY
    "k_-5": 8.2, # Wang, DeY
    "K_5": 0.082, # Wang, DeY - derived

    # Channel gains
    "k_IP3R": 5.55, #Wang
    "k_RyR": 5.0, #Wang 
    "k_ryr0": 0.0072, # Wang and Friel RyR opening rate
    "k_ryr1": 0.334, # Wang and Friel and Shannon RyR closing rate
    "k_ryr2": 0.5, # Wang and Friel and Shannon RyR activation affinity
    "k_ryr3": 38.0, # Wang and Shannon RyR inactivation affinity

    # Voltage parameters
    "c_m": 1.0, # Lata
    "I_stim": 0.1175, # Lata

    # Contraction
    "alpha": 3.0, #Lata - uterine
    "beta": 0.001, #Lata - uterine
    "n_F": 4, # Lata - uterine

    # Functional parameters
    "n": 4, # Wang - Hill coefficient for SERCA channel activation 1, 2 or 4
    "ns": 2, # Wang and Lytton
    "n2": 3, # Wang 3 or 5 

    "E": 5, 
    "k_e1": 0.01,
    "k_e2": 0.2, 
    "k_e_in": 0.9, 
    "h": 4, 
    "EC50": 2000
    }
# -----------------------------
# Flux definitions
# -----------------------------
def J_in1(V, Ca_in, Ca_0, p):
    return p["a_0"]-(p["a_1"]*I_Ca(V, Ca_in, Ca_0, p)/(2*p["F"])) * e_eff(p) +p["a_2"]*p["P"] 
def e_eff(p):
    return (1-p["k_e_in"]*p["E"])

def m_inf(V, p):
    return 1.0 / (1.0 + np.exp(-(V - p["V_m"]) / p["k_m"]))

def V_Ca(V, Ca_in, Ca_0, p):
    F = p["F"]
    R = p["R"]
    T = p["T"]

    exp_term = np.exp(-2.0 * V * F / (R * T))

    denom = 1.0 - exp_term
    if np.abs(denom) < 1e-8:
        return 1e-8

    return V * (Ca_in - Ca_0 * exp_term) / denom

def I_Ca(V, Ca_in, Ca_0, p):
    m = m_inf(V, p)
    Vca = V_Ca(V, Ca_in, Ca_0, p)
    return p["g_Ca"] * (m**2) * Vca

def J_PMCA_Hill(Ca_in, p):
    return p["V_Pmax"] * (Ca_in**p["n"]) / (p["k_PMCA"]**p["n"] + Ca_in**p["n"]) # - Ca_0**p["n"]

def J_SERCA_Hill(Ca_in, p):
    return p["V_s"] * (Ca_in**p["ns"]) / (p["k_s"]**p["ns"] + Ca_in**p["ns"]) # - Ca_0**p["n"]

def J_leak(Ca_SR, Ca_in, p):
    return p["k_leak"] * (Ca_SR - Ca_in)

def J_IP3R_Wang(Ca_SR, Ca_in, y_g, p):
    return p["k_IP3R"] * P_IP3R(Ca_in, y_g, p) * (Ca_SR - Ca_in)

def P_IP3R(Ca_in, y, p):
    num = p["P"] * Ca_in * (1 - y)
    den = (p["P"] + p["K_1"]) * (Ca_in + p["K_5"])
    return (num / den) ** 3

def dy_dt(y, p, Ca_in):
    f1 = (p["k_-4"] * p["K_2"] * p["K_1"] + p["k_-2"] * p["K_4"] * p["P"]) * Ca_in / (p["K_4"] * p["K_2"] * (p["K_1"] + p["P"]))
    f2 = (p["k_-2"] * p["P"] + p["k_-4"] * p["K_3"]) / (p["K_3"] + p["P"])
    return f1 * (1.0 - y) - f2 * y

def J_RyR_Hill(Ca_SR, Ca_in, p):
    return p["k_RyR"] * (Ca_SR**p["n"]/(p["K_RyR"]**p["n"]+Ca_SR**p["n"])) * (Ca_SR - Ca_in)

def J_RyR_Wang(Ca_SR, Ca_in, p):
    return p["k_RyR"] * P_RyR(Ca_in, Ca_SR, p) * (Ca_SR - Ca_in)

def P_RyR(Ca_in, Ca_SR, p):
    # CICR activation term (cytosolic Ca)
    activation = (
        p["k_ryr0"]
        + (p["k_ryr1"] * Ca_in**3) / (p["k_ryr2"]**3 + Ca_in**3)
    )

    # SR load dependence
    sr_term = Ca_SR**4 / (p["k_ryr3"]**4 + Ca_SR**4)

    return activation * sr_term

def residuals(x, p):
    """
    Steady-state equations for:
        x[0] = C_0
        x[1] = C_in
        x[2] = C_SR
    """
    C_0, C_in, C_SR, y = x
    V = -60

    Jin = p["delta"]*J_in1(V, C_in, C_0, p)
    Jpm = p["delta"]*J_PMCA_Hill(C_in, p)
    Jserca = J_SERCA_Hill(C_in, p)
    Jip3r = J_IP3R_Wang(C_SR, C_in, y, p)
    Jryr = J_RyR_Wang(C_SR, C_in, p)
    Jleak = J_leak(C_SR, C_in, p)

    F = np.zeros(4)
    F[0] = Jpm - Jin
    F[1] = Jin - Jpm - Jserca + Jip3r + Jryr + Jleak
    F[2] = p["gamma"]*(Jserca - Jip3r - Jryr - Jleak)
    f1 = (p["k_-4"] * p["K_2"] * p["K_1"] + p["k_-2"] * p["K_4"] * p["P"]) * C_in / (p["K_4"] * p["K_2"] * (p["K_1"] + p["P"]))
    f2 = (p["k_-2"] * p["P"] + p["k_-4"] * p["K_3"]) / (p["K_3"] + p["P"])
    F[3] = f1 * (1.0 - y) - f2 * y

    return F

def solve_with_root(x0, p):
    # sol = root(lambda x: residuals(x, p), x0, method="hybr")

    sol = least_squares(
        lambda x: residuals(x, params),
        x0,
        bounds=(0, np.inf)
        )
    print(sol.x)
    print(sol.fun)

    return {
        "x": sol.x,
        "success": sol.success,
        "message": sol.message,
        "fun": sol.fun,
        "nfev": sol.nfev,
        "status": sol.status,
    }

if __name__ == "__main__":
    params = get_default_params()

    x0 = np.array([1002, 0.1, 25.0, 0])
    out = solve_with_root(x0, params)

    print("Single solve")
    print("Success:", out["success"])
    print("Message:", out["message"])
    print("Steady state:", out["x"])
    print("Residuals:", out["fun"])
    print()

[1.05413503e+03 1.75399451e-01 1.06548854e+01 3.10237034e-01]
[ 1.26712009e-12 -1.24078525e-12 -1.44106949e-13 -5.74290615e-13]
Single solve
Success: True
Message: `gtol` termination condition is satisfied.
Steady state: [1.05413503e+03 1.75399451e-01 1.06548854e+01 3.10237034e-01]
Residuals: [ 1.26712009e-12 -1.24078525e-12 -1.44106949e-13 -5.74290615e-13]



# Adding back in the V equation - not correctly predicting the C_0 st st

In [ ]:
import numpy as np
from scipy.optimize import root, fsolve, least_squares

def get_default_params():
    return {
    # Calcium flux parameters
    "k_PMCA": 0.4,      # Wang extrusion rate 
    "V_Pmax": 4.5,      # Wang PMCA pump max rate 
    "V_s": 4.5,         # Wang/Lytton SR uptake rate max for SERCA pump between 0.1 and 0.5
    "k_s": 0.1,         # Wang/Lytton SR uptake affinity for SERCA pump between 0.1 and 0.5 
    "k_leak": 0.1,      # Wang SR leak
    "F": 96485.3329,    # Physical Faraday's constant 

    "P": 0.35, # IP3 production proxy by agonist 

    "gamma": 5.5,
    "delta": 0.05,

    "g_Ca": 9.0,        # Wang nS mM^-1 
    "V_m": -50.0,       # Wang mV 
    "k_m": 12.0,        # Wang mV 
    "R": 8314,         # Physical mJ/(mol K)
    "T": 310.0,          # Wang K (37°C) 

    "a_0": 0.05, # Wang
    "a_1": 0.25, # Wang
    "a_2": 1, # Wang

    "k_1": 2000, # Wang,. DeY 
    "k_-1": 260, # Wang, DeY
    "K_1": 0.13, # Wang, DeY - derived 
    "k_2": 1, # Wang, DeY
    "k_-2": 1.05, # Wang, DeY
    "K_2": 1.05, # Wang, DeY - derived
    "k_3": 2000, # Wang, DeY
    "k_-3": 1886, # Wang, DeY
    "K_3": 0.943, # Wang, DeY - derived
    "k_4": 1, # Wang, DeY
    "k_-4": 0.145, # Wang, DeY
    "K_4": 0.145, # Wang, DeY - derived
    "k_5": 100, # Wang, DeY
    "k_-5": 8.2, # Wang, DeY
    "K_5": 0.082, # Wang, DeY - derived

    # Channel gains
    "k_IP3R": 5.55, #Wang
    "k_RyR": 5.0, #Wang 
    "k_ryr0": 0.0072, # Wang and Friel RyR opening rate
    "k_ryr1": 0.334, # Wang and Friel and Shannon RyR closing rate
    "k_ryr2": 0.5, # Wang and Friel and Shannon RyR activation affinity
    "k_ryr3": 38.0, # Wang and Shannon RyR inactivation affinity

    # Voltage parameters
    "c_m": 1.0, # Lata
    "I_stim": 0.1175, # Lata

    # Contraction
    "alpha": 3.0, #Lata - uterine
    "beta": 0.001, #Lata - uterine
    "n_F": 4, # Lata - uterine

    # Functional parameters
    "n": 4, # Wang - Hill coefficient for SERCA channel activation 1, 2 or 4
    "ns": 2, # Wang and Lytton
    "n2": 3 # Wang 3 or 5 
    }
# -----------------------------
# Flux definitions
# -----------------------------

def J_in_Wang(V, Ca_in, Ca_0, p):
    return p["a_0"]-p["a_1"]*I_Ca(V, Ca_in, Ca_0, p)/(2*p["F"]) +p["a_2"]*p["P"]  

def m_inf(V, p):
    return 1.0 / (1.0 + np.exp(-(V - p["V_m"]) / p["k_m"]))

def V_Ca(V, Ca_in, Ca_0, p):
    F = p["F"]
    R = p["R"]
    T = p["T"]

    exp_term = np.exp(-2.0 * V * F / (R * T))

    denom = 1.0 - exp_term
    if np.abs(denom) < 1e-8:
        return 1e-8

    return V * (Ca_in - Ca_0 * exp_term) / denom

def I_Ca(V, Ca_in, Ca_0, p):
    m = m_inf(V, p)
    Vca = V_Ca(V, Ca_in, Ca_0, p)
    return p["g_Ca"] * (m**2) * Vca

def J_PMCA_Hill(Ca_in, p):
    return p["V_Pmax"] * (Ca_in**p["n"]) / (p["k_PMCA"]**p["n"] + Ca_in**p["n"]) # - Ca_0**p["n"]

def J_SERCA_Hill(Ca_in, p):
    return p["V_s"] * (Ca_in**p["ns"]) / (p["k_s"]**p["ns"] + Ca_in**p["ns"]) # - Ca_0**p["n"]

def J_leak(Ca_SR, Ca_in, p):
    return p["k_leak"] * (Ca_SR - Ca_in)

def J_IP3R_Wang(Ca_SR, Ca_in, y_g, p):
    return p["k_IP3R"] * P_IP3R(Ca_in, y_g, p) * (Ca_SR - Ca_in)

def P_IP3R(Ca_in, y, p):
    num = p["P"] * Ca_in * (1 - y)
    den = (p["P"] + p["K_1"]) * (Ca_in + p["K_5"])
    return (num / den) ** 3

def dy_dt(y, p, Ca_in):
    f1 = (p["k_-4"] * p["K_2"] * p["K_1"] + p["k_-2"] * p["K_4"] * p["P"]) * Ca_in / (p["K_4"] * p["K_2"] * (p["K_1"] + p["P"]))
    f2 = (p["k_-2"] * p["P"] + p["k_-4"] * p["K_3"]) / (p["K_3"] + p["P"])
    return f1 * (1.0 - y) - f2 * y

def J_RyR_Hill(Ca_SR, Ca_in, p):
    return p["k_RyR"] * (Ca_SR**p["n"]/(p["K_RyR"]**p["n"]+Ca_SR**p["n"])) * (Ca_SR - Ca_in)

def J_RyR_Wang(Ca_SR, Ca_in, p):
    return p["k_RyR"] * P_RyR(Ca_in, Ca_SR, p) * (Ca_SR - Ca_in)

def P_RyR(Ca_in, Ca_SR, p):
    # CICR activation term (cytosolic Ca)
    activation = (
        p["k_ryr0"]
        + (p["k_ryr1"] * Ca_in**3) / (p["k_ryr2"]**3 + Ca_in**3)
    )

    # SR load dependence
    sr_term = Ca_SR**4 / (p["k_ryr3"]**4 + Ca_SR**4)

    return activation * sr_term

def residuals(x, p):
    """
    Steady-state equations for:
        x[0] = C_0
        x[1] = C_in
        x[2] = C_SR
    """
    C_0, C_in, C_SR, V, y = x

    Jin = p["delta"]*J_in_Wang(V, C_in, C_0, p)
    Jpm = p["delta"]*J_PMCA_Hill(C_in, p)
    Jserca = J_SERCA_Hill(C_in, p)
    Jip3r = J_IP3R_Wang(C_SR, C_in, y, p)
    Jryr = J_RyR_Wang(C_SR, C_in, p)
    Jleak = J_leak(C_SR, C_in, p)

    F = np.zeros(5)
    F[0] = Jpm - Jin
    F[1] = Jin - Jpm - Jserca + Jip3r + Jryr + Jleak
    F[2] = p["gamma"]*(Jserca - Jip3r - Jryr - Jleak)
    # Jserca - Jip3r - Jryr - Jleak
    F[3] = (C_0 - C_in) / p["c_m"]
    # F[3] = ((p["k_-4"] * p["K_2"] * p["K_1"] + p["k_-2"] * p["K_4"] * p["P"]) * C_in / (p["K_4"] * p["K_2"] * (p["K_1"] + p["P"])))/((p["k_-4"] * p["K_2"] * p["K_1"] + p["k_-2"] * p["K_4"] * p["P"]) * C_in / (p["K_4"] * p["K_2"] * (p["K_1"] + p["P"]))+(p["k_-2"] * p["P"] + p["k_-4"] * p["K_3"]) / (p["K_3"] + p["P"]))
    f1 = (p["k_-4"] * p["K_2"] * p["K_1"] + p["k_-2"] * p["K_4"] * p["P"]) * C_in / (p["K_4"] * p["K_2"] * (p["K_1"] + p["P"]))
    f2 = (p["k_-2"] * p["P"] + p["k_-4"] * p["K_3"]) / (p["K_3"] + p["P"])
    F[4] = f1 * (1.0 - y) - f2 * y

    return F

def solve_with_root(x0, p):
    # sol = root(lambda x: residuals(x, p), x0, method="hybr")    

    lower_bounds = [0.0, 0.0, 0.0, -100.0, 0.0]
    upper_bounds = [2000.0, 10.0, 100.0, 100.0, 1.0]

    sol = least_squares(
        lambda x: residuals(x, params),
        x0,
        bounds=(lower_bounds, np.inf)
        )
    print(sol.x)
    print(sol.fun)

    return {
        "x": sol.x,
        "success": sol.success,
        "message": sol.message,
        "fun": sol.fun,
        "nfev": sol.nfev,
        "status": sol.status,
    }


if __name__ == "__main__":
    params = get_default_params()

    x0 = np.array([1002, 0.1, 25.0, -60, 0])
    out = solve_with_root(x0, params)

    print("Single solve")
    print("Success:", out["success"])
    print("Message:", out["message"])
    print("Steady state:", out["x"])
    print("Residuals:", out["fun"])
    print()

[  0.22356034   0.22356034  12.01114491 -26.23807104   0.36438177]
[ 7.04231361e-11 -7.15660864e-11  6.28574970e-12  1.66533454e-15
 -6.23681939e-11]
Single solve
Success: True
Message: `gtol` termination condition is satisfied.
Steady state: [  0.22356034   0.22356034  12.01114491 -26.23807104   0.36438177]
Residuals: [ 7.04231361e-11 -7.15660864e-11  6.28574970e-12  1.66533454e-15
 -6.23681939e-11]

